<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"></ul></div>

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

In [2]:
data = pd.read_csv('Train.csv')
data

,ID,Class
0,0.jpg,MIDDLE
1,1.jpg,OLD
2,3.jpg,MIDDLE
3,4.jpg,MIDDLE
4,5.jpg,YOUNG
...,...,...
495,645.jpg,YOUNG
496,646.jpg,MIDDLE
497,648.jpg,YOUNG
498,649.jpg,MIDDLE


In [3]:
data['Class'].unique()

array(['MIDDLE', 'OLD', 'YOUNG'], dtype=object)

In [4]:
data['Class'].replace({'MIDDLE':1, 'OLD':2, 'YOUNG':0},inplace=True)

C:\Users\saiar\AppData\Local\Temp\ipykernel_4192\2713183201.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Class'].replace({'MIDDLE':1, 'OLD':2, 'YOUNG':0},inplace=True)
C:\Users\saiar\AppData\Local\Temp\ipykernel_4192\2713183201.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data['C

In [5]:
data

,ID,Class
0,0.jpg,1
1,1.jpg,2
2,3.jpg,1
3,4.jpg,1
4,5.jpg,0
...,...,...
495,645.jpg,0
496,646.jpg,1
497,648.jpg,0
498,649.jpg,1


In [6]:
def read_image(path,ch=3,resize=(150,150)):
    di = tf.io.read_file(path)
    di = tf.image.decode_jpeg(di,channels=ch)
    di = tf.image.convert_image_dtype(di,dtype=tf.float32)
    di = tf.image.resize(di,resize)
    return di

In [7]:
def load_data(image_path,label):
    img = read_image(image_path,3,(150,150))
    return img,label

In [8]:
p = pd.DataFrame(os.listdir('Train'),columns=['paths'])

In [9]:
p

,paths
0,0.jpg
1,1.jpg
2,100.jpg
3,101.jpg
4,102.jpg
...,...
495,95.jpg
496,96.jpg
497,97.jpg
498,98.jpg


In [10]:
p = p.sample(100)

In [11]:
p

,paths
437,611.jpg
423,599.jpg
58,167.jpg
240,389.jpg
379,554.jpg
...,...
204,35.jpg
63,172.jpg
462,644.jpg
417,593.jpg


In [12]:
Path = 'Train'
image_paths = []

for path in p['paths']:
    image_paths.append(Path+'/'+path)

In [13]:
len(image_paths)

100

In [14]:
image_paths[0:50]

['Train/611.jpg',
 'Train/599.jpg',
 'Train/167.jpg',
 'Train/389.jpg',
 'Train/554.jpg',
 'Train/250.jpg',
 'Train/247.jpg',
 'Train/12.jpg',
 'Train/22.jpg',
 'Train/249.jpg',
 'Train/268.jpg',
 'Train/76.jpg',
 'Train/112.jpg',
 'Train/370.jpg',
 'Train/94.jpg',
 'Train/441.jpg',
 'Train/19.jpg',
 'Train/390.jpg',
 'Train/519.jpg',
 'Train/413.jpg',
 'Train/615.jpg',
 'Train/45.jpg',
 'Train/538.jpg',
 'Train/404.jpg',
 'Train/237.jpg',
 'Train/329.jpg',
 'Train/65.jpg',
 'Train/85.jpg',
 'Train/204.jpg',
 'Train/93.jpg',
 'Train/496.jpg',
 'Train/451.jpg',
 'Train/57.jpg',
 'Train/580.jpg',
 'Train/149.jpg',
 'Train/223.jpg',
 'Train/264.jpg',
 'Train/424.jpg',
 'Train/120.jpg',
 'Train/63.jpg',
 'Train/115.jpg',
 'Train/442.jpg',
 'Train/359.jpg',
 'Train/330.jpg',
 'Train/558.jpg',
 'Train/108.jpg',
 'Train/282.jpg',
 'Train/592.jpg',
 'Train/134.jpg',
 'Train/236.jpg']

In [15]:
os.path.split('C/users/sai_files/Train/0.jpg')

('C/users/sai_files/Train', '0.jpg')

In [16]:
a,b,c = (1,2,3)

In [11]:
b

2

In [23]:
data[data['ID']=='0.jpg']['Class'].values[0]

np.int64(1)

In [41]:
data.loc[data['ID']==tail]['Class'].values[0]

1

In [19]:
data

,ID,Class
264,351.jpg,1
101,127.jpg,1
355,467.jpg,1
437,574.jpg,1
324,426.jpg,1
...,...,...
135,172.jpg,1
399,527.jpg,1
370,486.jpg,0
35,48.jpg,0


In [18]:
data.loc[data['ID']=='0.jpg']['Class']

Series([], Name: Class, dtype: int64)

In [24]:
response_list = []
for i in image_paths:
    _,tail = os.path.split(i)
    response = data.loc[data['ID']==tail]['Class'].values[0]
    response_list.append(response)

In [25]:
len(response_list)

100

In [29]:
train_size = int(0.9*len(image_paths))
test_size = int(0.1*len(image_paths))

In [30]:
test_size

10

In [31]:
train_size

90

In [32]:
train_set = tf.data.Dataset.from_tensor_slices((image_paths[:train_size],response_list[:train_size]))
test_set = tf.data.Dataset.from_tensor_slices((image_paths[train_size:],response_list[train_size:]))

In [21]:
train_set = (train_set
            .map(load_data,num_parallel_calls=tf.data.AUTOTUNE)
            .batch(64)
            .prefetch(tf.data.AUTOTUNE))

In [22]:
test_set = (test_set
            .map(load_data,num_parallel_calls=tf.data.AUTOTUNE)
            .batch(64)
            .prefetch(tf.data.AUTOTUNE))

In [23]:
from tensorflow.keras import Sequential,layers,models

In [24]:
cnn_model = Sequential([
    layers.Conv2D(filters=30,kernel_size=(3,3),activation='relu',input_shape=(150,150,3)),
    layers.MaxPool2D((2,2)),
    
    layers.Conv2D(filters=64,kernel_size=(3,3),activation='relu'),
    layers.MaxPool2D((2,2)),
    layers.Flatten(),
    layers.Dense(64,activation='relu'),
    layers.Dense(3,activation='softmax')
])

In [25]:
cnn_model.compile(loss='sparse_categorical_crossentropy',
                 optimizer = 'adam',
                 metrics=['accuracy'])

In [26]:
cnn_model.fit(train_set,epochs=3,validation_data=test_set)

Epoch 1/3
2/2 [==============================] - 4s 803ms/step - loss: 1.9805 - accuracy: 0.2556 - val_loss: 5.7016 - val_accuracy: 0.3000
Epoch 2/3
2/2 [==============================] - 2s 594ms/step - loss: 3.5745 - accuracy: 0.3667 - val_loss: 1.3983 - val_accuracy: 0.6000
Epoch 3/3
2/2 [==============================] - 2s 549ms/step - loss: 1.0074 - accuracy: 0.6111 - val_loss: 1.1980 - val_accuracy: 0.5000


In [27]:
cnn_model.evaluate(test_set)

1/1 [==============================] - 0s 107ms/step - loss: 1.1980 - accuracy: 0.5000


[1.1980422735214233, 0.5]

In [37]:
cnn_model.save('age_pred.h5')

C:\Users\saiar\AppData\Local\anaconda3\Lib\site-packages\keras\src\engine\training.py:3079: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
